# 🚦 Lane Logic — Traffic Demand Prediction
## Gridlock Hackathon 2.0 | Flipkart × Bengaluru Traffic Police

| | |
|---|---|
| **Team** | Lane Logic |
| **Participant** | Neha Binu |
| **Problem** | Predict traffic demand (0–1) for 41,778 road segments in Bengaluru on Day 49 |
| **Metric** | `score = max(0, 100 × R²(actual, predicted))` |
| **Final Score** | **90.79 / 100** |

---

## The Core Insight

Bengaluru traffic follows **daily habit patterns**. MG Road at 10am on Day 48 looks almost identical to MG Road at 10am on Day 49. This gave us our golden feature: `demand_lag1day` — same zone, same timestamp, previous day. This single feature had a **0.79 correlation** with the target, explaining ~79% of all variance.


## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
import lightgbm as lgb
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

print(f"Train: {train.shape} | Test: {test.shape}")
print(f"Target range: {train['demand'].min():.4f} to {train['demand'].max():.4f}")
print(f"Train columns: {list(train.columns)}")


## 2. Exploratory Data Analysis

In [ ]:
# Understanding the temporal structure — this is critical
print("Days in train:", sorted(train['day'].unique()))
print("Days in test: ", sorted(test['day'].unique()))

d48 = train[train['day']==48]
d49 = train[train['day']==49]
print(f"\nDay 48 rows: {len(d48)} | timestamps: {d48['timestamp'].nunique()} (full day)")
print(f"Day 49 rows: {len(d49)} | timestamps: {sorted(d49['timestamp'].unique())}")
print("\n=> Train Day 49 covers only midnight to 2am (quiet hours)")
print("=> Test covers 2:15am onwards — we predict using Day 48 as the lag signal")


In [ ]:
# Key signals in the data
print("Mean demand by RoadType:")
print(train.groupby('RoadType')['demand'].mean().sort_values(ascending=False).round(4))

print("\nMean demand by NumberofLanes:")
print(train.groupby('NumberofLanes')['demand'].mean().sort_values(ascending=False).round(4))

print("\nTop 5 zones by mean demand (Day 48):")
print(d48.groupby('geohash')['demand'].mean().sort_values(ascending=False).head().round(4))


In [ ]:
# Verify the lag correlation — the foundation of our model
d48_lag = d48[['geohash','timestamp','demand']].rename(columns={'demand':'lag'})
d49_check = d49.merge(d48_lag, on=['geohash','timestamp'], how='left')
corr = d49_check[['demand','lag']].corr().iloc[0,1]
print(f"Correlation (demand vs demand_lag1day): {corr:.4f}")
print("=> 0.79 — yesterday at this exact location and time predicts today strongly")


## 3. Feature Engineering

### What we built and why

| Feature | What it captures |
|---|---|
| `demand_lag_best` | Precise match: same zone + time + road type + lanes from Day 48 |
| `last_d49` | Most recent demand today (2:00am Day 49) — same-day signal |
| `geo_hr_mean` | How busy this zone typically is at this hour |
| `geo_road_hr` | Zone × road type × hour — very specific pattern |
| `road_mean/road_max` | Stats per geohash × RoadType × Lanes |
| `prefix_mean` | Neighbourhood average (first 4 chars of geohash) |
| `hw_peak` | Highway during peak hours — strong interaction |
| `lane_dens` | Geo mean demand ÷ lanes — congestion pressure |


In [ ]:
def ts_to_minutes(ts):
    """Convert '10:15' -> 615 (minutes since midnight)"""
    h, m = ts.split(':')
    return int(h) * 60 + int(m)

def engineer_features(df, train_df):
    """
    Full feature engineering pipeline.
    train_df is always the full training set — used to compute statistics
    from Day 48 only to prevent data leakage into predictions.
    """
    df = df.copy()
    d48 = train_df[train_df['day']==48]

    # ── Time features ────────────────────────────────────────────────────────
    df['minutes']        = df['timestamp'].apply(ts_to_minutes)
    df['hour']           = df['minutes'] // 60
    df['minute_of_hour'] = df['minutes'] % 60
    df['is_peak']        = df['hour'].isin([8,9,10,11,12,13,14,17,18,19,20]).astype(int)
    df['is_night']       = df['hour'].isin([0,1,2,3,4,5]).astype(int)
    df['is_morning']     = df['hour'].isin([6,7,8,9]).astype(int)
    df['is_evening']     = df['hour'].isin([17,18,19,20,21]).astype(int)
    df['hour_sin']       = np.sin(2 * np.pi * df['hour'] / 24)  # Circular encoding
    df['hour_cos']       = np.cos(2 * np.pi * df['hour'] / 24)

    # ── Precise lag: 4-key match ─────────────────────────────────────────────
    # KEY DISCOVERY: each geohash zone has multiple road segments.
    # Matching on geohash+timestamp+RoadType+Lanes gives the EXACT same road yesterday.
    # This is more accurate than averaging all segments in a zone.
    precise_lag = (d48[['geohash','timestamp','RoadType','NumberofLanes','demand']]
                   .rename(columns={'demand':'demand_precise_lag'}))
    df = df.merge(precise_lag, on=['geohash','timestamp','RoadType','NumberofLanes'], how='left')

    # ── Fuzzy lag fallback: 2-key match (zone average) ──────────────────────
    fuzzy_lag = (d48.groupby(['geohash','timestamp'])['demand']
                 .mean().reset_index().rename(columns={'demand':'demand_lag1day'}))
    df = df.merge(fuzzy_lag, on=['geohash','timestamp'], how='left')

    # Use precise where available, fuzzy as fallback
    df['demand_lag_best'] = df['demand_precise_lag'].fillna(df['demand_lag1day'])

    # ── Road-type statistics ─────────────────────────────────────────────────
    road_stats = (d48.groupby(['geohash','RoadType','NumberofLanes'])['demand']
                  .agg(road_mean='mean', road_max='max').reset_index())
    df = df.merge(road_stats, on=['geohash','RoadType','NumberofLanes'], how='left')

    # ── Geohash-level statistics ─────────────────────────────────────────────
    geo_stats = (d48.groupby('geohash')['demand']
                 .agg(geo_mean='mean', geo_std='std', geo_max='max',
                      geo_min='min', geo_median='median').reset_index())
    df = df.merge(geo_stats, on='geohash', how='left')

    # ── Geohash × hour ───────────────────────────────────────────────────────
    geo_hour = d48.copy()
    geo_hour['hour'] = geo_hour['timestamp'].apply(lambda x: int(x.split(':')[0]))
    df = df.merge(geo_hour.groupby(['geohash','hour'])['demand'].mean()
                  .reset_index().rename(columns={'demand':'geo_hr_mean'}),
                  on=['geohash','hour'], how='left')

    # ── Geohash × RoadType × hour ────────────────────────────────────────────
    geo_road_hour = d48.copy()
    geo_road_hour['hour'] = geo_road_hour['timestamp'].apply(lambda x: int(x.split(':')[0]))
    df = df.merge(geo_road_hour.groupby(['geohash','RoadType','hour'])['demand'].mean()
                  .reset_index().rename(columns={'demand':'geo_road_hr'}),
                  on=['geohash','RoadType','hour'], how='left')

    # ── Global timestamp stats ───────────────────────────────────────────────
    ts_stats = (d48.groupby('timestamp')['demand']
                .agg(ts_mean='mean', ts_med='median').reset_index())
    df = df.merge(ts_stats, on='timestamp', how='left')

    # ── Same-day last known demand (Day 49 train, up to 2:00am) ────────────
    d49_tr = train_df[train_df['day']==49].copy()
    d49_tr['m'] = d49_tr['timestamp'].apply(ts_to_minutes)
    df = df.merge(d49_tr.sort_values('m').groupby('geohash')['demand'].last()
                  .reset_index().rename(columns={'demand':'last_d49'}),
                  on='geohash', how='left')

    # ── Neighbourhood cluster (geohash prefix = ~1km area) ──────────────────
    df['geo_prefix'] = df['geohash'].str[:4]
    px = d48.copy(); px['geo_prefix'] = px['geohash'].str[:4]
    df = df.merge(px.groupby('geo_prefix')['demand'].mean()
                  .reset_index().rename(columns={'demand':'prefix_mean'}),
                  on='geo_prefix', how='left')

    # ── Categorical encoding ─────────────────────────────────────────────────
    df['RoadType']  = df['RoadType'].fillna('Unknown')
    df['RT_enc']    = df['RoadType'].map({'Highway':3,'Street':2,'Residential':1,'Unknown':0}).fillna(0)
    df['LV_enc']    = (df['LargeVehicles']=='Allowed').astype(int)
    df['LM_enc']    = (df['Landmarks']=='Yes').astype(int)
    df['WX_enc']    = df['Weather'].map({'Sunny':0,'Rainy':1,'Foggy':2,'Snowy':3}).fillna(-1)

    # ── Interaction features ─────────────────────────────────────────────────
    df['hw_flag']   = (df['RoadType']=='Highway').astype(int)
    df['hw_peak']   = df['hw_flag'] * df['is_peak']           # Highway at peak = most congested
    df['lane_dens'] = df['geo_mean'] / df['NumberofLanes'].replace(0,1)  # Congestion per lane
    df['lag_xlane'] = df['demand_lag_best'] * df['NumberofLanes']
    df['lag_xroad'] = df['demand_lag_best'] * df['RT_enc']

    # ── Temperature imputation by timestamp median ───────────────────────────
    tt = train_df.groupby('timestamp')['Temperature'].median().to_dict()
    df['Temperature'] = df['Temperature'].fillna(df['timestamp'].map(tt))
    df['Temperature'] = df['Temperature'].fillna(train_df['Temperature'].median())

    # ── Geohash label encode ─────────────────────────────────────────────────
    df['geo_enc'] = df['geohash'].map(
        {g:i for i,g in enumerate(sorted(train_df['geohash'].unique()))}).fillna(-1).astype(int)
    df['geo_cl']  = df['geo_prefix'].map(
        {g:i for i,g in enumerate(sorted(df['geo_prefix'].unique()))}).fillna(-1).astype(int)

    # ── Fill remaining missing values ────────────────────────────────────────
    df['demand_lag_best'] = df['demand_lag_best'].fillna(df['road_mean']).fillna(df['geo_mean']).fillna(df['ts_mean'])
    df['last_d49']        = df['last_d49'].fillna(df['geo_mean'])
    df['road_mean']       = df['road_mean'].fillna(df['geo_mean'])
    df['road_max']        = df['road_max'].fillna(df['geo_max'])
    df['geo_hr_mean']     = df['geo_hr_mean'].fillna(df['geo_mean'])
    df['geo_road_hr']     = df['geo_road_hr'].fillna(df['geo_hr_mean'])
    df['prefix_mean']     = df['prefix_mean'].fillna(df['geo_mean'])
    for c in ['geo_mean','geo_std','geo_max','geo_min','geo_median','ts_mean','ts_med']:
        df[c] = df[c].fillna(df[c].median())

    return df.fillna(0)

print("Building features...")
train_feat = engineer_features(train, train)
test_feat  = engineer_features(test,  train)
print(f"Done. Train: {train_feat.shape} | Test: {test_feat.shape}")


## 4. Modelling — LightGBM + CatBoost Ensemble

In [ ]:
FEATURES = [
    'minutes','hour','minute_of_hour','is_peak','is_night','is_morning','is_evening',
    'hour_sin','hour_cos',
    'demand_lag_best',            # Golden feature: same zone+time+road yesterday
    'last_d49',                   # Most recent demand today (2am Day 49)
    'road_mean','road_max',       # Zone × road type stats
    'geo_road_hr',                # Zone × road type × hour
    'geo_mean','geo_std','geo_max','geo_min','geo_median',
    'geo_hr_mean',                # Zone × hour average
    'ts_mean','ts_med',           # City-wide pattern at this time
    'prefix_mean',                # Neighbourhood average
    'RT_enc','NumberofLanes','LV_enc','LM_enc','WX_enc','Temperature',
    'hw_flag','hw_peak',          # Highway interactions
    'lane_dens','lag_xlane','lag_xroad',
    'geo_enc','geo_cl','day'
]

X      = train_feat[FEATURES]
y      = train_feat['demand']
X_test = test_feat[FEATURES]

print(f"Feature count : {len(FEATURES)}")
print(f"Training shape: {X.shape}")
print(f"Missing values: train={X.isnull().sum().sum()} | test={X_test.isnull().sum().sum()}")


In [ ]:
kf       = KFold(n_splits=5, shuffle=True, random_state=42)
oof_lgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))
test_lgb = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))

# ── LightGBM ─────────────────────────────────────────────────────────────────
# Fast gradient boosting — industry standard for tabular data competitions
LGB_PARAMS = dict(
    objective='regression', metric='rmse',
    num_leaves=255, learning_rate=0.05,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
    min_child_samples=20, n_estimators=1500,
    random_state=42, verbose=-1
)

print("Training LightGBM (5-fold CV)...")
for fold, (tri, vli) in enumerate(kf.split(X)):
    m = lgb.LGBMRegressor(**LGB_PARAMS)
    m.fit(X.iloc[tri], y.iloc[tri],
          eval_set=[(X.iloc[vli], y.iloc[vli])],
          callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(False)])
    p = m.predict(X.iloc[vli])
    oof_lgb[vli] = p
    test_lgb    += m.predict(X_test) / 5
    print(f"  Fold {fold+1}: {max(0, 100*r2_score(y.iloc[vli], p)):.4f}")
print(f"  LightGBM OOF Score: {max(0, 100*r2_score(y, oof_lgb)):.4f}")


In [ ]:
# ── CatBoost ─────────────────────────────────────────────────────────────────
# Handles categorical features natively — no encoding needed internally
# Often outperforms LightGBM when categoricals are important (RoadType, Weather)
print("Training CatBoost (5-fold CV)...")
for fold, (tri, vli) in enumerate(kf.split(X)):
    m = CatBoostRegressor(
        iterations=1000, learning_rate=0.05, depth=8,
        loss_function='RMSE', random_seed=42, verbose=0
    )
    m.fit(X.iloc[tri], y.iloc[tri],
          eval_set=(X.iloc[vli], y.iloc[vli]),
          early_stopping_rounds=50)
    p = m.predict(X.iloc[vli])
    oof_cat[vli] = p
    test_cat    += m.predict(X_test) / 5
    print(f"  Fold {fold+1}: {max(0, 100*r2_score(y.iloc[vli], p)):.4f}")
print(f"  CatBoost OOF Score:  {max(0, 100*r2_score(y, oof_cat)):.4f}")


In [ ]:
# ── Find optimal blend weights ───────────────────────────────────────────────
# Blending reduces individual model errors — if LGB is wrong on one row,
# CatBoost often compensates, and vice versa
print("Blend weight search:")
best_score, best_w = 0, 0.4
for w in [0.2, 0.3, 0.4, 0.5, 0.6]:
    blend_oof = w * oof_lgb + (1-w) * oof_cat
    s = max(0, 100 * r2_score(y, blend_oof))
    marker = " ← best" if s > best_score else ""
    print(f"  LGB {w:.0%} + CAT {1-w:.0%}: {s:.4f}{marker}")
    if s > best_score:
        best_score, best_w = s, w

print(f"\nOptimal: LGB {best_w:.0%} + CAT {1-best_w:.0%} = {best_score:.4f}")


## 5. Generate Submission

In [ ]:
# Blend and clip to valid demand range [0, 1]
final_preds = np.clip(best_w * test_lgb + (1 - best_w) * test_cat, 0, 1)

submission = pd.DataFrame({
    'Index':  test['Index'],
    'demand': final_preds
})

submission.to_csv('submission_final.csv', index=False)
print(f"Submission saved: {submission.shape}")
print(f"Demand range: {submission['demand'].min():.4f} – {submission['demand'].max():.4f}")
print(submission.head(10))


## 6. Score Progression

| Version | Key Changes | Online Score |
|---------|------------|-------------|
| v1 | Baseline: demand_lag1day + geo stats + LightGBM 5-fold | 90.15 |
| v2 | + last_known_d49, smarter lag imputation, demand trend | 90.54 |
| v4 | + precise 4-key lag matching, road×hour, geo×road×hour, interaction features | 90.607 |
| **v5** | **+ CatBoost blend (40% LGB + 60% CAT)** | **90.79** |

Each version improved through understanding the data better — not random tuning.

## 7. Key Learnings

1. **Temporal data needs temporal thinking.** Random KFold inflated CV to 99.3 but real performance was 90.79. Proper validation must respect time order — train on the past, validate on the future.

2. **Feature engineering > model selection.** The biggest single improvement came from realising each geohash zone has multiple road segments and matching the lag on all 4 keys instead of averaging.

3. **Blending works.** CatBoost handles categoricals natively and made different errors than LightGBM. Together they beat either model alone.

4. **Domain knowledge is a feature.** Knowing that Bengaluru traffic follows daily habits led directly to `demand_lag1day` — the most important feature in the model, contributing 79% of explained variance.
